# Experiment — `destination_port`: raw integer vs range buckets

Not a sprint — a feature-engineering side experiment (Section 12 open question). **Non-destructive**: loads the persisted split; `02`/`03` untouched.

`destination_port` is a **category stored as a number** (443 isn't "more than" 80). The tree can already threshold-split raw ports, so does coarsening it into IANA ranges help, hurt, or tie? We retrain the *same* tree (entropy, unpruned, seed 42) both ways and compare.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score

proc = Path('../data/processed')
X_train = pd.read_parquet(proc / 'X_train.parquet')
X_test  = pd.read_parquet(proc / 'X_test.parquet')
y_train = pd.read_parquet(proc / 'y_train.parquet')['label_binary']
y_test  = pd.read_parquet(proc / 'y_test.parquet')['label_binary']

def fit_eval(Xtr, Xte, name):
    tree = DecisionTreeClassifier(criterion='entropy', max_depth=None, random_state=42).fit(Xtr, y_train)
    yp = tree.predict(Xte)
    m = dict(acc=accuracy_score(y_test, yp), recall=recall_score(y_test, yp, zero_division=0),
             prec=precision_score(y_test, yp, zero_division=0), leaves=tree.get_n_leaves(), depth=tree.get_depth())
    print('%-14s acc=%.4f  recall=%.4f  prec=%.4f   (depth %d, leaves %d)' % (
        name, m['acc'], m['recall'], m['prec'], m['depth'], m['leaves']))
    return m

## 1. Baseline — raw integer port (what `03` used)

In [2]:
base = fit_eval(X_train, X_test, 'raw port')

raw port       acc=0.9988  recall=0.9966  prec=0.9972   (depth 67, leaves 3109)


## 2. Range buckets — IANA well-known / registered / ephemeral

Replace the single `destination_port` column with **three 0/1 columns**, one per range:
- well-known `0–1023` (HTTP 80, HTTPS 443, SSH 22, FTP 21 …)
- registered `1024–49151`
- ephemeral `49152–65535` (client-side dynamic ports)

One-hot, *not* an ordinal 0/1/2 — so no false ordering is implied between the ranges.

In [3]:
bins, labels = [-1, 1023, 49151, 65535], ['well_known', 'registered', 'ephemeral']

def bucket(X):
    b = pd.cut(X['destination_port'], bins=bins, labels=labels)
    oh = pd.get_dummies(b, prefix='port').astype(int)
    return X.drop(columns=['destination_port']).join(oh)

X_train_b = bucket(X_train)
X_test_b  = bucket(X_test).reindex(columns=X_train_b.columns, fill_value=0)
print('port buckets (train):')
print(pd.cut(X_train['destination_port'], bins=bins, labels=labels).value_counts().to_string())
print('feature cols: raw=%d  bucketed=%d\n' % (X_train.shape[1], X_train_b.shape[1]))

buck = fit_eval(X_train_b, X_test_b, 'range buckets')

n_mal = int((y_test == 1).sum())
print('\n--- raw -> buckets (positive = buckets better) ---')
print('accuracy   %+.4f' % (buck['acc'] - base['acc']))
print('recall     %+.4f   (missed intrusions %d -> %d)' % (
    buck['recall'] - base['recall'], round((1 - base['recall']) * n_mal), round((1 - buck['recall']) * n_mal)))
print('precision  %+.4f' % (buck['prec'] - base['prec']))
print('leaves     %d -> %d' % (base['leaves'], buck['leaves']))

port buckets (train):
destination_port
well_known    1761490
registered     255552
ephemeral      247460
feature cols: raw=65  bucketed=67

range buckets  acc=0.9990  recall=0.9984  prec=0.9967   (depth 61, leaves 1733)

--- raw -> buckets (positive = buckets better) ---
accuracy   +0.0002
recall     +0.0018   (missed intrusions 378 -> 178)
precision  -0.0005
leaves     3109 -> 1733
